# GPT-5.4 Prompting Guide

## 1. Introduction

GPT-5.4 is a frontier model in the GPT-5 family intended for complex professional workloads such as coding, document analysis, structured extraction, research synthesis, and multi-step agent workflows. It builds on prompting patterns that already work well for GPT-5 and GPT-5.2, while benefiting from newer platform capabilities for longer workflows, stronger tool integration, and more operationally structured execution.

In practice, most prompt structures that work well for GPT-5.2 continue to work well for GPT-5.4. The main difference is not that GPT-5.4 requires an entirely new prompting style; it is that clearer scope, completion criteria, verification rules, and tool-use contracts tend to matter more as tasks become longer, more tool-heavy, and more production-critical.

This guide focuses on practical prompt patterns for developers building with GPT-5.4 in production. It emphasizes explicit task boundaries, predictable output contracts, grounded tool usage, and eval-driven iteration.

> **Scope note**
> This guide is a practical companion to official GPT-5.4 API documentation and GPT-5-family prompting guidance. Where behavior is directly documented, this guide states it directly. Where recommendations are extrapolated from GPT-5-class prompting patterns or platform capabilities, they are presented as practical patterns to validate with evals before production standardization.

## 2. Key behavioral differences

Compared with earlier GPT-5 models, GPT-5.4 is especially useful in workflows that involve:

* longer contexts
* stronger integration with tools and external systems
* multi-step task execution with verification requirements
* structured outputs consumed by downstream systems

These characteristics change what matters most in prompts.

### 2.1 Long-context workflows

GPT-5-class models can operate over very large contexts, which makes them useful for:

* multi-document reasoning
* long conversation histories
* long-running coding and agent workflows
* retrieval-augmented document analysis

In these settings, prompts benefit from explicit re-grounding. When answers depend on specific clauses, sections, dates, or thresholds, the prompt should tell the model to anchor claims to the relevant source locations rather than summarize generically.

```text
<long_context_handling>
- For large inputs, first identify the sections most relevant to the user’s goal.
- Re-state key constraints before answering (e.g. jurisdiction, date range, product, team, file set).
- Anchor claims to sections, documents, pages, or file names rather than speaking generically.
- If the answer depends on fine details, quote or paraphrase the relevant details instead of compressing them away.
</long_context_handling>
```

### 2.2 Tool-integrated execution

Many GPT-5.4 deployments rely on tools such as:

* search APIs
* retrieval systems
* internal databases
* code execution environments
* office/document processing pipelines

Prompts should therefore define when the model should rely on tools rather than internal reasoning. This is especially important when facts are fresh, user-specific, or operationally important.

### 2.3 Structured multi-step execution

GPT-5.4 is often used inside workflows that plan, retrieve information, make intermediate decisions, and then produce a final output. In these cases, prompts are usually more reliable when they specify:

* what counts as completion
* which steps are in scope
* when updates are needed
* how to verify that a result is correct

This guide focuses on prompt patterns that support those workflows without assuming undocumented model internals.

## 3. How to choose the right prompt pattern

A strong prompt guide should help you choose patterns, not just collect them. Use the following selection rules as a starting point.

| Workflow type                 | Highest-leverage prompt patterns                                                                   |
| ----------------------------- | -------------------------------------------------------------------------------------------------- |
| Extraction / document parsing | Schema definition, missing-data rules, source anchoring, re-scan step                              |
| Tool-heavy agent workflow     | Completion criteria, tool-use rules, prerequisite checks, verification after writes                |
| Long-context reasoning        | Re-grounding, source anchoring, explicit constraint restatement                                    |
| High-risk output              | Ambiguity handling, assumption labeling, self-check, evidence requirements                         |
| Prompt migration              | Freeze prompt first, pin `reasoning_effort`, run evals, then tighten only where regressions appear |
| Research / synthesis          | Source quality rules, contradiction handling, citation contract, research-depth contract           |

A useful rule of thumb: start with the smallest prompt that can plausibly succeed, then add only the constraints required to fix observed failures.

## 4. Prompting patterns

The patterns below improve reliability across many GPT-5-class workflows. They are not magic phrases. They are structured prompt design practices that make instructions easier for the model to follow consistently.

### 4.1 Controlling verbosity and output shape

Explicit output constraints reduce unnecessary verbosity and improve formatting consistency.

Example:

```text
<output_verbosity_spec>
- Default: 3–6 sentences or ≤5 bullets.
- For simple factual questions: ≤2 sentences.
- For complex multi-step or multi-file tasks:
  - 1 short overview paragraph
  - then ≤5 bullets tagged: What changed, Where, Risks, Next steps, Open questions.
- Avoid repeating the user’s request unless needed for disambiguation.
- Prefer concise explanations over long narrative paragraphs.
- Use bullets, short sections, and tables when they improve scanability.
</output_verbosity_spec>
```

This pattern is particularly useful when responses must fit downstream systems such as APIs, dashboards, structured reports, or workflow logs.

### 4.2 Preventing scope drift

Models capable of complex reasoning may expand the task unless prompts define boundaries explicitly.

Example:

```text
<scope_constraints>
- Implement exactly what the user requested.
- Do not introduce additional features, analyses, or formatting flourishes unless explicitly requested.
- Preserve existing interfaces, names, and structure unless the task asks for changes.
- If multiple interpretations exist, choose the simplest one consistent with the request.
- If you notice adjacent work, call it out as optional instead of doing it automatically.
</scope_constraints>
```

Scope constraints are especially valuable in coding, spreadsheet editing, reporting, and document transformation workflows where “helpful expansion” can become drift.

### 4.3 Instruction ordering and role definition

Complex prompts usually work better when instructions are presented in priority order.

Example:

```text
<role_and_priority>
You are an implementation assistant for technical workflows.

Follow instructions in this order:
1. Satisfy the user’s explicit request.
2. Follow required output format and schema.
3. Use tools or provided context when needed.
4. Avoid unnecessary commentary or extra work.
5. Ask clarifying questions only if essential information is missing and cannot be retrieved.
</role_and_priority>
```

When a prompt contains many rules, global behavior rules should usually come before task-specific requirements, and output formatting rules should usually come after both.

A practical prompt layout is often:

1. role and priority order
2. global operating rules
3. task-specific instructions
4. output contract
5. verification or self-check rules

### 4.4 Handling ambiguity and hallucination risk

When a prompt lacks critical information, the model may infer details incorrectly unless the prompt defines how ambiguity should be handled.

Mitigation pattern:

```text
<ambiguity_handling>
- If the request is ambiguous or underspecified, identify the missing information.
- Ask concise clarifying questions when necessary, OR proceed with clearly labeled assumptions if the workflow requires forward progress.
- Do not invent exact numbers, line references, citations, policies, or IDs.
- If assumptions are required, state them explicitly.
- If external facts may have changed and no tools are available, answer in general terms and qualify the uncertainty.
</ambiguity_handling>
```

For higher-risk contexts, add a short self-check step:

```text
<high_risk_self_check>
Before finalizing in legal, financial, compliance, or safety-sensitive contexts:
- Re-scan the answer for unsupported assumptions.
- Re-scan for specific claims not grounded in the prompt or tool outputs.
- Soften overly strong language where certainty is not justified.
</high_risk_self_check>
```

### 4.5 Completion criteria

A surprising number of prompt failures are not reasoning failures. They are definition-of-done failures. Make completion criteria explicit whenever outputs have multiple parts, downstream consumers, or verification requirements.

Example:

```text
<completion_criteria>
A task is complete only when:
- all requested deliverables are present,
- required schema or format is satisfied,
- tool-derived claims are grounded in retrieved results,
- unresolved ambiguity is either clarified or labeled as an assumption,
- and any required verification steps have been performed.
</completion_criteria>
```

Completion criteria are especially useful in agents, extraction pipelines, and multi-step operational workflows.

### 4.6 Prompt minimization

Longer prompts are not automatically better prompts. Overgrown prompts often accumulate stale rules, hidden contradictions, and ceremonial fluff.

Recommended practice:

* start with the smallest prompt that can succeed
* add constraints only when a specific failure appears
* prefer a few high-leverage rules over many overlapping ones
* remove stale instructions after workflow changes
* test whether each added instruction measurably improves evals

This is how you avoid building a majestic cathedral of conflicting instructions and then acting surprised when the gargoyles start lying.

## 5. Before/after examples

The fastest way to improve prompt quality is often to compare a vague prompt with a tighter one.

### 5.1 Extraction example

**Weak prompt**

```text
Summarize these contracts and tell me the risks.
```

**Why it fails**

* no output shape
* no definition of risk
* no source anchoring
* no missing-data rules

**Better prompt**

```text
Review these contracts and produce:
1. a 1-paragraph executive summary,
2. a bullet list of material risks,
3. a JSON array with:
   - document_id
   - governing_law
   - termination_for_convenience
   - payment_terms
   - risk_level

Rules:
- Anchor each risk to a contract section or clause.
- If a field is missing, use null.
- Do not infer governing law unless explicitly stated.
- A task is complete only when every contract has both prose summary output and JSON output.
```

### 5.2 Code modification example

**Weak prompt**

```text
Improve this component.
```

**Why it fails**

* no scope boundary
* no success criteria
* invites feature creep
* unclear whether behavior, style, performance, or maintainability should change

**Better prompt**

```text
Modify this component to fix the loading-state layout bug.

Constraints:
- Preserve existing props and exported names.
- Do not change styling except where necessary to fix the layout issue.
- Do not add new features.
- Return:
  1. the updated code,
  2. a short explanation of what changed,
  3. any risks or follow-up checks.
- The task is complete only when the loading state matches the normal layout structure.
```

### 5.3 Research example

**Weak prompt**

```text
Research the best database for analytics.
```

**Why it fails**

* “best” is underspecified
* no research depth defined
* no comparison dimensions
* no evidence standard

**Better prompt**

```text
Compare analytics databases for a mid-size SaaS company.

Requirements:
- Compare at least 4 credible options.
- Evaluate cost model, query latency, operational complexity, concurrency, and ecosystem support.
- Prefer primary or official sources where possible.
- Cite factual claims.
- If tradeoffs differ by workload, label the workload assumptions clearly.
- End with a recommendation section that distinguishes “best for low ops burden,” “best for performance,” and “best for cost control.”
```

## 6. Compaction (Extending Effective Context)

Long-running workflows may eventually exceed practical context limits if conversation state keeps growing. GPT-5-class systems support conversation compaction, which compresses earlier conversation state into a smaller representation intended to preserve task-relevant information while reducing context usage.

Typical use cases include:

* long research workflows
* agent systems with many tool calls
* extended document analysis sessions
* long-running coding or migration tasks

Compaction is usually applied after major milestones rather than after every message.

### When to use compaction

* after a large retrieval phase
* after a planning phase that produced durable constraints
* after a long tool-heavy execution phase
* when conversation state is growing faster than task value

### Best practices

* compact after milestones, not every turn
* keep prompts functionally consistent before and after compaction
* re-state key user constraints after compaction in high-stakes workflows
* treat compacted outputs as opaque system state rather than human-readable summaries

Example:



In [ ]:
from openai import OpenAI
import json

client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input=[{
        "role": "user",
        "content": "Draft a migration plan for a legacy billing system."
    }]
)

output_json = [item.model_dump() for item in response.output]

compacted_response = client.responses.compact(
    model="gpt-5.4",
    input=[
        {
            "role": "user",
            "content": "Draft a migration plan for a legacy billing system."
        },
        output_json[0]
    ]
)

print(json.dumps(compacted_response.model_dump(), indent=2))



## 7. Agentic steerability and user updates

Multi-step workflows benefit from structured progress updates and explicit plan-revision rules.

Example:

```text
<user_updates_spec>
- Provide brief updates only when:
  - a new major phase of work begins,
  - a key discovery changes the plan,
  - or user confirmation is required.
- Each update should include one concrete outcome.
- Do not narrate routine low-value steps.
- Do not expand the task beyond the user’s request; surface adjacent work as optional.
</user_updates_spec>
```

Planning rules can also improve long-running task stability:

```text
<plan_revision_rules>
- Begin with a short plan for multi-step tasks.
- Revise the plan only when new evidence changes priorities, scope, or sequencing.
- Preserve completed work whenever possible.
- If the plan changes, briefly explain what changed and why.
</plan_revision_rules>
```

These patterns improve operator visibility while limiting low-value narration.

## 8. Tool-calling, dependency checks, and stopping rules

Tool use improves reliability when tasks require fresh data, external state, or system-grounded results.

Best practices include:

* describing each tool clearly
* preferring tools when data may be outdated or user-specific
* verifying state-changing operations
* parallelizing only independent retrieval steps
* checking prerequisites before downstream actions

Example:

```text
<tool_usage_rules>
- Prefer tools when fresh, external, or user-specific data is required.
- Parallelize independent read operations where possible.
- Before any write operation, confirm parameters and target state.
- After writes, verify the resulting state.
- If a tool result conflicts with prior assumptions, trust the tool result and revise the plan.
</tool_usage_rules>
```

For more complex tool workflows, add dependency-aware behavior:

```text
<dependency_checks>
- Before taking an action, verify whether prerequisite retrieval or lookup is required.
- Do not skip prerequisite steps just because the final action seems obvious.
- If a retrieval step returns empty or partial results, try a fallback strategy before concluding failure.
</dependency_checks>
```

Stopping rules are also important. Many tool-using agents fail by either stopping too early or thrashing forever.

```text
<tool_stopping_rules>
- Continue retrieval until the required fields, evidence, or verification targets are satisfied.
- Stop when additional tool use is unlikely to change the answer materially.
- Do not repeat equivalent searches without a new hypothesis or fallback strategy.
</tool_stopping_rules>
```

### What to validate in evals

When testing tool-heavy prompts, look for:

* premature stopping after partial tool use
* missing prerequisite lookups
* incorrect parallelization of dependent steps
* write actions without read-back verification
* excessive tool churn without better answers

## 9. Structured extraction, PDF, and Office workflows

Structured extraction tasks benefit from explicit schemas, clear field requirements, and explicit handling of missing data.

Example:

```text
<extraction_spec>
Extract structured data into JSON.

Schema:
{
  "document_id": string,
  "counterparty": string | null,
  "effective_date": string | null,
  "financial_amount": string | null,
  "source_reference": string | null
}

Rules:
- Follow the schema exactly.
- Use null for missing values.
- Do not guess missing data.
- Preserve original units, dates, and currencies as written.
- Before returning, re-scan once for missed fields or schema mismatches.
</extraction_spec>
```

Providing an explicit schema substantially improves extraction consistency.

For multi-file workflows, serialize one output per document and include a stable identifier so downstream systems can reconcile outputs reliably.

## 10. Prompt Migration Guide to GPT-5.4

When migrating prompts from earlier models, the safest approach is incremental evaluation.

### Migration mapping

| Current model | Target model | Target reasoning_effort          | Notes                                                                                                            |
| ------------- | ------------ | -------------------------------- | ---------------------------------------------------------------------------------------------------------------- |
| GPT-4o        | GPT-5.4      | none                             | Start with the fastest setting and add structure before increasing reasoning effort.                             |
| GPT-5         | GPT-5.4      | same value except minimal → none | Preserve the prior latency/depth profile first, then evaluate changes.                                           |
| GPT-5.2       | GPT-5.4      | same value                       | Most GPT-5.2 prompts should migrate cleanly with only targeted tightening for scope, verification, or structure. |

### Recommended workflow

1. Switch models without modifying the prompt.
2. Pin the `reasoning_effort` parameter explicitly.
3. Run evaluation tests to establish a baseline.
4. Adjust prompt structure only if regressions appear.
5. Re-evaluate after each change.

This process isolates the effect of each modification and reduces accidental prompt drift.

### What to validate in evals

When migrating prompts, measure:

* requirement satisfaction
* structure adherence
* latency and token usage
* tool behavior
* citation integrity for research workflows

## 11. Evaluation framework

“Run evals” is correct advice, but it is too vague on its own. Useful evals should test the exact ways a prompt can fail in production.

### Suggested eval dimensions

| Dimension                | What to measure                                                    | Typical regression signal                        |
| ------------------------ | ------------------------------------------------------------------ | ------------------------------------------------ |
| Requirement satisfaction | Whether the user’s actual request was completed                    | Missing deliverables or partial completion       |
| Structure adherence      | Whether the required format or schema was followed                 | Extra fields, wrong ordering, format drift       |
| Factual grounding        | Whether material claims are supported                              | Unsupported specificity or invented details      |
| Citation integrity       | Whether claims are properly attributed                             | Missing, misplaced, or fabricated citations      |
| Tool selection quality   | Whether the right tools were used at the right times               | Tool avoidance or unnecessary tool use           |
| Tool-call efficiency     | Whether the workflow converges efficiently                         | Repeated equivalent lookups or tool thrash       |
| Verification behavior    | Whether writes and high-risk outputs were validated                | Missing read-back or unverified state changes    |
| Latency and token usage  | Whether the prompt stays operationally efficient                   | Cost spikes, excessive reasoning, slow responses |
| Recoverability           | Whether the system handles ambiguity or partial failure gracefully | Fragile behavior after incomplete retrieval      |

Representative test cases matter more than pretty demo prompts. Easy evals flatter bad prompts.

## 12. Web search and research

Research workflows benefit from explicit evidence requirements and output contracts.

Best practices include:

* defining research depth up front
* requiring citations for material factual claims
* preferring primary or official sources where possible
* instructing the model how to handle ambiguity or conflicting evidence

Example:

```text
<web_research_rules>
- Base claims on reliable sources.
- Prefer primary or official references where possible.
- Cite sources for factual statements.
- If sources disagree, explain the discrepancy.
- Match the depth of research to the task.
- If ambiguity remains, either ask a concise clarifying question or proceed with labeled assumptions.
</web_research_rules>
```

For deeper research workflows, add a stronger contract:

```text
<research_depth_contract>
- Quick lookup: verify the key fact with a small number of high-quality sources.
- Medium research: compare the main dimensions and resolve obvious contradictions.
- Deep research: follow second-order leads until more search is unlikely to change the answer.
- Do not fabricate citations or imply verification you did not perform.
</research_depth_contract>
```

Source quality rules also help:

```text
<source_quality_rules>
- Prefer primary sources for technical, legal, scientific, financial, and product claims.
- Use secondary sources for synthesis and context.
- Use multiple sources when claims are disputed, recent, or consequential.
- Distinguish official claims from independent verification when both matter.
</source_quality_rules>
```

Structured research prompts improve traceability and reduce unsupported synthesis.

## 13. Prompt stacks for common workflows

The patterns above are ingredients. In production, you often want assembled prompt stacks.

### 13.1 Extraction agent

```text
<role_and_priority>
You are a structured extraction assistant.
Follow instructions in this order:
1. Extract only what is supported by the source.
2. Follow the schema exactly.
3. Use null for missing fields.
4. Anchor important fields to source references.
5. Re-scan once before returning.
</role_and_priority>

<extraction_spec>
Extract structured data into the required JSON schema.
Do not add fields.
Preserve units, dates, and currencies as written.
</extraction_spec>

<completion_criteria>
The task is complete only when every required field has a valid value or null, the schema matches exactly, and the output includes one record per source document.
</completion_criteria>
```

### 13.2 Code modification agent

```text
<role_and_priority>
You are a code modification assistant.
Priority order:
1. Satisfy the requested change.
2. Preserve existing interfaces unless the task requires changes.
3. Avoid unrelated edits.
4. Return a concise change summary and any relevant risks.
</role_and_priority>

<scope_constraints>
- Implement exactly the requested fix.
- Do not add features.
- Keep naming and structure stable unless the task requires otherwise.
</scope_constraints>

<completion_criteria>
The task is complete only when the requested behavior is implemented, unrelated behavior is preserved, and the response includes both code and a concise explanation.
</completion_criteria>
```

### 13.3 Research agent

```text
<role_and_priority>
You are a rigorous research assistant.
Priority order:
1. Answer the user’s actual question.
2. Base material claims on evidence.
3. Use reliable sources.
4. Label uncertainty and assumptions clearly.
</role_and_priority>

<web_research_rules>
- Prefer primary or official sources where possible.
- Cite factual claims.
- If sources disagree, explain the disagreement.
- Match research depth to the stakes and complexity of the task.
</web_research_rules>

<research_depth_contract>
- Quick lookup: verify the key fact.
- Medium research: compare major dimensions.
- Deep research: follow second-order leads until further search is unlikely to change the answer materially.
</research_depth_contract>
```

### 13.4 Operations workflow agent

```text
<tool_usage_rules>
- Prefer tools for fresh, external, or user-specific data.
- Check prerequisites before downstream actions.
- Before writes, confirm parameters and target state.
- After writes, verify the resulting state.
</tool_usage_rules>

<user_updates_spec>
- Provide brief updates only when a new major phase begins or a discovery changes the plan.
- Include one concrete outcome in each update.
</user_updates_spec>

<completion_criteria>
The task is complete only when all requested actions have been performed, any required state changes have been verified, and any unresolved issues are clearly surfaced.
</completion_criteria>
```

## 14. Common failure modes and remedies

A good guide should not only show best practices. It should also show where prompts go off the rails.

| Failure mode           | Typical symptom                                           | Prompt remedy                                                   |
| ---------------------- | --------------------------------------------------------- | --------------------------------------------------------------- |
| Scope drift            | Adds helpful but unrequested work                         | Add explicit scope constraints and optional-work handling       |
| Format drift           | Mostly follows structure, then adds extra prose or fields | Add a stricter output contract and completion criteria          |
| Premature closure      | Stops after partial evidence or retrieval                 | Add stopping rules and completion criteria                      |
| Tool avoidance         | Answers from prior knowledge when tools were needed       | Add explicit tool-use rules tied to freshness or external state |
| Tool thrashing         | Repeats similar lookups without converging                | Add stopping rules and fallback strategy rules                  |
| Ungrounded specificity | Invents exact values, IDs, citations, or line references  | Add ambiguity handling and anti-fabrication rules               |
| Long-context blur      | Gives generic summary instead of source-anchored answer   | Add re-grounding and source-anchoring rules                     |
| Verification theater   | Claims something was checked without real validation      | Require read-back verification or explicit validation steps     |
| Instruction collision  | Conflicting rules produce unstable behavior               | Simplify prompt and separate global rules from local rules      |

## 15. Handling ambiguity without unnecessary questions

Not all ambiguity should trigger a clarifying question. A useful prompt decides how to move forward based on the kind of missing information.

| Situation                                                           | Best response                             |
| ------------------------------------------------------------------- | ----------------------------------------- |
| Missing information blocks correctness entirely                     | Ask a concise clarifying question         |
| Missing information affects optimization, style, or preference only | Proceed with a clearly labeled assumption |
| Multiple plausible interpretations materially change the answer     | Cover the top interpretations explicitly  |

This is one of the simplest ways to reduce both hallucination and needless back-and-forth.

## 16. Avoid contradictory prompt requirements

Some prompt failures are self-inflicted. The prompt asks the model to do incompatible things and then acts shocked when the output gets weird.

Common collisions include:

* “be exhaustive” and “answer in two sentences”
* “never ask questions” and “do not make assumptions”
* “use only the provided context” and “include up-to-date information”
* “preserve exact structure” and “improve formatting as needed”

Recommended practice:

* decide which instruction has priority
* explicitly state tradeoffs when constraints conflict
* remove rules that are no longer aligned with the workflow

## 17. Debugging prompt behavior

When prompt behavior is inconsistent, the most useful debugging steps are often the simplest.

* **Check output contracts first.** Many failures occur because the requested structure is underspecified.
* **Reduce instruction conflicts.** Very long prompts with mixed priorities can create hidden contradictions.
* **Check tool requirements.** If the model avoids tools, the prompt may not state clearly when tool use is required.
* **Test on representative inputs.** Prompt behavior often looks better on easy examples than on real production distributions.
* **Tighten prompts before raising reasoning effort.** In many cases, clearer scope and better verification outperform simply increasing reasoning depth.
* **Check whether completion criteria are missing.** If outputs are partial, the model may not know what “done” means.
* **Check whether the prompt is trying to do too many jobs at once.** Separate research, transformation, and decision-making stages when useful.

## 18. Production prompt checklist

Before shipping a prompt, check the following:

* is scope explicit?
* is the output contract concrete?
* are missing-data rules defined?
* does the prompt specify when tools are required?
* are completion criteria stated?
* are verification rules included for writes or high-risk outputs?
* are ambiguity rules defined?
* have evals been run on realistic cases?
* is any instruction redundant, stale, or conflicting?

## 19. Conclusion

GPT-5.4 continues the GPT-5 family’s emphasis on structured reasoning, tool integration, and multi-step task execution.

Effective prompting typically involves:

* explicit scope definition
* structured output contracts
* grounded tool usage
* ambiguity handling for underspecified tasks
* explicit completion criteria
* evaluation-driven iteration

Teams that combine clear prompts with systematic testing generally achieve the most reliable results.

## Limitations

This guide does not claim undocumented model internals or training details. Some recommendations are derived from general GPT-5 prompting practices and GPT-5.4-era platform capabilities, and may vary depending on the application.

Production systems should validate prompt behavior using representative evaluation tasks before adopting any pattern at scale.

## Appendix

### Appendix A: Example prompt for a web research agent

```text
You are a rigorous research assistant.

CORE MISSION
- Answer the user’s question directly and completely.
- Base material claims on evidence.
- If something cannot be verified, say so clearly.
- Add adjacent context only when it materially helps the user.

RESEARCH RULES
- Prefer reliable and primary sources when possible.
- Use web research when facts may be incomplete, recent, or disputed.
- Follow important second-order leads for deep research tasks.
- If ambiguity remains, either ask a concise clarifying question or proceed with clearly labeled assumptions.

CITATION RULES
- Cite factual claims derived from sources.
- Do not fabricate citations, links, or quoted text.
- If sources disagree, explain the disagreement.

OUTPUT STYLE
- Use Markdown with clear headers and bullets.
- Start with the direct answer.
- Then provide supporting evidence, caveats, and recommended next steps when useful.

FINAL CHECK
Before answering, verify that:
- material claims are supported,
- contradictions are addressed,
- assumptions are labeled,
- and the response matches the requested depth.
```

### Appendix B: End-to-end worked example — long-context document review

**Task**
Review a set of policy documents and identify data-retention obligations.

**Prompt**

```text
You are a policy review assistant.

Follow instructions in this order:
1. Identify all data-retention obligations relevant to the user’s request.
2. Anchor each finding to the source document and section.
3. Do not infer retention periods unless explicitly stated.
4. If a policy is ambiguous, label the ambiguity clearly.

Return:
- a short summary,
- a table with document name, section, retention requirement, and ambiguity notes,
- a short list of conflicts or missing information.

A task is complete only when every provided document has been checked and all retention-related findings are source-anchored.
```

**Why this works**

* it defines scope tightly
* it requires source anchoring
* it defines the deliverables
* it prevents fabricated retention periods
* it states what counts as completion

### Appendix C: End-to-end worked example — spreadsheet or reporting workflow

**Task**
Update a recurring KPI report without changing its structure.

**Prompt**

```text
You are a reporting workflow assistant.

Constraints:
- Update the report with the new input data.
- Preserve the existing sheet structure, formulas, labels, and ordering unless a change is explicitly requested.
- Flag missing or inconsistent data instead of guessing.
- Return a concise summary of what changed and any validation checks performed.

Completion criteria:
- all updated values are reflected in the report,
- existing formulas and layout are preserved,
- missing data is surfaced explicitly,
- and the summary states any risks or unresolved issues.
```

**Why this works**

* it prevents scope drift
* it protects structure
* it forces explicit handling of incomplete data
* it adds validation expectations
